# 📋 [OPS08_M09] 피드백 반영 역할별 보고 및 사격 절차 재훈련을 통한 반응 속도 개선
### AI Co-pilot 협업 기반 소부대 대공 방어 표준운용절차(SOP) 및 모의훈련 워크북 (학생 실습용 워크북)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS08/ops08_m09_student.ipynb)

---

## 📌 프로젝트 개요
- **과정명:** 드론 경보 시 관측자-지휘자-경계조 역할분담 모의훈련 (Level 2)
- **미션 번호:** OPS08_M09
- **미션 주제:** 피드백 반영 역할별 보고 및 사격 절차 재훈련을 통한 반응 속도 개선
- **운용 원칙:**
  1. **HITL 원칙:** AI 제안은 참고 지표이며, 최종 지휘·승인·교전 판단은 인간이 확정한다.
  2. **군사보안 준수:** 실제 부대명, 좌표, 통신 음어, 무기 제원은 일체 사용하지 않으며 가상 시뮬레이션 데이터만 활용한다.
  3. **안전 제일 원칙:** 실사격/장비 조작은 배제하며 구두 선언과 모의 기동으로만 진행하고 교관 지휘에 엄격히 따른다.


In [ ]:
# [OPS08] 소부대 드론 경보 모의훈련 자동 분석 및 검증 헬퍼 모듈
import pandas as pd
import numpy as np

def verify_mission1_raci(raci_data, checklist_items):
    """미션 1 RACI 매트릭스 및 체크리스트 무결성 검증"""
    print("=" * 60)
    print("📋 [미션 1 검증] 7단계 역할분담 매트릭스 & 체크리스트 검증")
    print("=" * 60)
    
    stages = ["포착", "식별", "보고", "전파", "결심", "대피·대응", "해제"]
    stage_pass = True
    for s in stages:
        if s not in raci_data:
            print(f"❌ 누락된 단계: {s}")
            stage_pass = False
        else:
            roles = raci_data[s]
            acc_count = sum(1 for r, duty in roles.items() if "책" in duty or "A" in duty)
            if acc_count != 1:
                print(f"⚠️ [{s} 단계] 책임자(A/책)가 정확히 1명이 아닙니다! (현재 {acc_count}명)")
                stage_pass = False
    
    check_count = len(checklist_items)
    check_pass = check_count >= 15
    print(f"✔ 현장 체크리스트 항목 수: {check_count}개 / 기준 15개 이상 {'[PASS]' if check_pass else '[FAIL]'}")
    print(f"✔ 7단계 RACI 매트릭스 책임자 무결성: {'[PASS]' if stage_pass else '[FAIL]'}")
    
    if stage_pass and check_pass:
        print("\n🎉 [미션 1 PASS] 현장 점검 기준과 역할 분담이 전술 기준에 완벽히 부합합니다!")
    else:
        print("\n❌ [보완 필요] 누락된 단계나 중복된 책임자를 점검하십시오.")

def analyze_mission4_dryrun(log_df):
    """미션 4 1차 모의훈련 타임라인 및 준수율 분석"""
    print("=" * 60)
    print("⏱️ [미션 4 분석] 1차 Dry-run 훈련 타임라인 및 SOP 준수율 계산")
    print("=" * 60)
    
    avg_lead_time = log_df['lead_time_sec'].mean()
    sop_pass_rate = (log_df['sop_pass'].sum() / len(log_df)) * 100
    
    print(log_df[['event_id', 'detect_sec', 'report_sec', 'relay_sec', 'action_sec', 'lead_time_sec', 'sop_pass']])
    print("-" * 60)
    print(f"✔ 평균 조치 완료 시간: {avg_lead_time:.1f}초 (기준: 180초 이내) -> {'[PASS]' if avg_lead_time <= 180 else '[FAIL]'}")
    print(f"✔ SOP 단계별 준수율: {sop_pass_rate:.1f}% (기준: 80% 이상) -> {'[PASS]' if sop_pass_rate >= 80 else '[FAIL]'}")
    
    if avg_lead_time <= 180 and sop_pass_rate >= 80:
        print("\n🎉 [미션 4 PASS] 1차 Dry-run 목표치를 달성하였습니다! 병목 구간 2건을 AAR에 인계하십시오.")

def grade_mission5_cross_verification(user_answers, answer_key):
    """미션 5 교차확인 판정 채점"""
    print("=" * 60)
    print("🎯 [미션 5 채점] 오류 주입 교차확인 판정 결과 검증")
    print("=" * 60)
    
    correct = 0
    miss_threat = 0
    total = len(answer_key)
    
    for card, info in answer_key.items():
        user_pred = user_answers.get(card, {}).get("pred")
        truth = info["truth"]
        is_threat = ("실제 위협" in truth)
        
        is_correct = (user_pred == truth)
        if is_correct:
            correct += 1
        elif is_threat and ("오경보" in str(user_pred)):
            miss_threat += 1
            print(f"🚨 [치명적 오류] {card} 위협을 오경보로 놓쳤습니다! (Miss 발생)")
            
    success_rate = (correct / total) * 100
    print(f"✔ 허위 경보 식별 성공률: {success_rate:.1f}% ({correct}/{total}) (기준: 85% 이상)")
    print(f"✔ 실제 위협 미탐(Miss) 건수: {miss_threat}건 (기준: 0건)")
    
    if success_rate >= 85.0 and miss_threat == 0:
        print("\n🎉 [미션 5 PASS] 교차 확인을 통해 적 기만 및 허위 경보를 완벽히 필터링했습니다!")

def compare_mission9_retrial(m4_data, m9_data):
    """미션 9 1차 대비 2차 훈련 개선율 분석"""
    print("=" * 60)
    print("📈 [미션 9 성과 분석] 1차(M4) 대비 2차 재훈련 반응 속도 & 오류 개선율")
    print("=" * 60)
    
    time1 = m4_data['avg_time']
    time2 = m9_data['avg_time']
    time_reduction = ((time1 - time2) / time1) * 100
    
    err1 = m4_data['errors']
    err2 = m9_data['errors']
    err_reduction = ((err1 - err2) / err1) * 100
    
    print(f"1. 전체 조치 소요 시간: {time1:.1f}초 ➔ {time2:.1f}초 (단축률: {time_reduction:.1f}% / 목표 30% 이상)")
    print(f"2. SOP 절차 오류 건수: {err1}건 ➔ {err2}건 (감소율: {err_reduction:.1f}% / 목표 50% 이상)")
    print("-" * 60)
    
    pass_time = time_reduction >= 30.0
    pass_err = err_reduction >= 50.0
    
    if pass_time and pass_err:
        print("🎉 [미션 9 PASS] 수정 SOP 적용을 통해 팀 단위 전술 반응 속도가 입증되었습니다!")


---

## 🎯 미션 09. 피드백 반영 역할별 보고 및 사격 절차 재훈련을 통한 반응 속도 개선

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"동일 조건 하 2차 재훈련 데이터를 1차와 일대일 대조하여 조치 시간 단축률(30% 목표)과 오류 감소율(50% 목표)을 계산하고 보수 교육 로드맵을 작성해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 성과 향상이 단순 암기 효과가 아닌 절차 숙달에 의한 것인지 동일 측정 기준으로 검증합니다.

---

### 📑 실습자료 및 과업 지침

피드백 반영 역할별 보고 및 사격 절차 재훈련을 통한 반응 속도 개선**

가. 제공 자료 - 재훈련 동일 조건 유지표

| **항목** | **1차(M4)** | **재훈련** | **동일 여부** |
| --- | --- | --- | --- |
| **주입 카드 구성** | 10장 | 같은 10장(순서만 변경) | ( ) |
| **역할 편성** |  |  | ( ) |
| **평가표** | M4 서식 다 | M4 서식 다 | ( ) |
| **측정 방법** | 초시계·시각 기록표 |  | ( ) |
| **시간대·환경** |  |  | ( ) |

나. 제공 자료 - 계산 예시 [교육용가상자료]

| 평균 조치 시간: 1차 210초 → 재훈련 140초
단축률 = (210 - 140) ÷ 210 × 100 = 33.3% → 목표 30% 이상 달성

오류 건수: 1차 12건 → 재훈련 5건
감소율 = (12 - 5) ÷ 12 × 100 = 58.3% → 목표 50% 이상 달성 |
| --- |

다. 작성 서식 1 - 전후 성과 비교표

| **지표** | **1차(M4)** | **재훈련** | **변화율(%)** | **목표** | **달성** |
| --- | --- | --- | --- | --- | --- |
| **평균 조치 시간(초)** |  |  |  | 30% 이상 단축 | ( ) |
| **평균 보고 지연(초)** |  |  |  | 참고 | ( ) |
| **SOP 준수율(%)** |  |  |  | 참고 | ( ) |
| **오류 건수(누락)** |  |  |  | - | - |
| **오류 건수(오보고)** |  |  |  | - | - |
| **오류 건수(절차 위반)** |  |  |  | - | - |
| **오류 합계** |  |  |  | 50% 이상 감소 | ( ) |

라. 작성 서식 2 - 미달 항목 원인 및 보수 교육 계획

| **미달 항목** | **원인** | **보완 방법** |
| --- | --- | --- |
| **미달 1** |  |  |
| **미달 2** |  |  |

| **보수 교육 주기** | ( )주마다 ( )분, 방법: ( ) |
| --- | --- |

| **숙련 유지 확인 지표** | ( ) ※ 예: 평균 보고 지연 60초 이내 유지 |
| --- | --- |

| **점검 항목** | **확인** |
| --- | --- |
| 1차와 같은 카드·평가표·측정 방법을 썼다 | ( ) |
| 조치 시간 단축률을 계산식대로 계산했다 | ( ) |
| 오류를 유형별로 세고 감소율을 계산했다 | ( ) |
| 미달 항목의 원인을 적었다 | ( ) |
| 보수 교육 주기를 제안했다 | ( ) |

**



In [ ]:
# [미션 9 실습 코드] 1차 vs 2차 모의훈련 측정 데이터 입력
# TODO: 1차 훈련과 2차 재훈련의 평균 시간과 오류 발생 건수를 입력하고 개선율을 산출하세요.
m4_data = {'avg_time': 185.0, 'errors': 8}  # 1차 성과
m9_data = {'avg_time': 0.0, 'errors': 0}    # TODO: 2차 측정값을 입력하세요
# compare_mission9_retrial(m4_data, m9_data)
